In [12]:
import os 
import pandas as pd 
# Ar에 대한 데이터
final_adsorption_data_273K = pd.read_csv("./final_adsorption_data_273K.csv")
final_adsorption_data_313K = pd.read_csv("./final_adsorption_data_313K.csv")

EX = pd.read_csv("./Ar_273K/Ar_273K_Ar_273_0.01_to_Ar_273_15_dataset.csv")
EX.columns

META =  pd.read_csv("../2019-11-01-ASR-public_12020.csv")[['filename', 'LCD', 'PLD', 'LFPD', 'cm3_g', 'ASA_m2_cm3', 'ASA_m2_g',
       'NASA_m2_cm3', 'NASA_m2_g', 'AV_VF', 'AV_cm3_g', 'NAV_cm3_g',
        'Has_OMS',]]
# META의 Has_OMS 라벨인코당
META['Has_OMS'] = META['Has_OMS'].map({'Yes':1, 'No':0})
META
# EX.columns , META.columns
# (Index(['filename', 'LCD', 'PLD', 'LFPD', 'cm3_g', 'ASA_m2_cm3', 'ASA_m2_g',
#         'NASA_m2_cm3', 'NASA_m2_g', 'AV_VF', 'AV_cm3_g', 'NAV_cm3_g', 'Has_OMS',
#         'name', 'Input', 'Output'],
#        dtype='object'),
#  Index(['filename', 'LCD', 'PLD', 'LFPD', 'cm3_g', 'ASA_m2_cm3', 'ASA_m2_g',
#         'NASA_m2_cm3', 'NASA_m2_g', 'AV_VF', 'AV_cm3_g', 'NAV_cm3_g',
#         'Has_OMS'],
#        dtype='object'))
final_adsorption_data_273K.columns , final_adsorption_data_313K.columns
# (Index(['name', '0.01', '0.05', '0.1', '0.5', '1.0', '5.0', '15.0',
#         'henry_coeff'],
#        dtype='object'),
#  Index(['name', '0.01', '0.05', '0.1', '0.5', '1.0', '5.0', '15.0',
#         'henry_coeff'],
#        dtype='object'))''

# 저압 리스트 Henry , 0.01 , 0.05 , 0.1 , 0.5
# 고압 리스트 1, 5, 15
# 이때 273K, 313K 따로따로 수행하는데 이때 EX처럼 Input은 저압리스트로 Output은 고압리스트로 맞춰줘야함
# 조합이 콤비네이션으로 존재하고 완성되면 EX의 csv이름처럼 만들어줘야함.
# 이때 저장되는 폴더는 Ar_NEW_온도 로 ㄱㄱ
# -------------------------------------------------------------------------
# 아래부터 추가 코드입니다.
# -------------------------------------------------------------------------

# 0. 기본 설정 및 전처리
# META 데이터의 'filename' 컬럼을 흡착 데이터와 병합하기 위해 'name'으로 변경
META.rename(columns={'filename': 'name'}, inplace=True)

# 압력 구간 리스트 정의 (컬럼 이름이 문자열이므로 문자열로 정의)
low_pressure_list = ['henry_coeff', '0.01', '0.05', '0.1', '0.5']
high_pressure_list = ['1.0', '5.0', '15.0']

# 처리할 온도별 데이터와 정보를 딕셔너리로 묶어주면 코드를 재사용하기 편리합니다.
data_to_process = {
    '273K': final_adsorption_data_273K,
    '313K': final_adsorption_data_313K
}

# 1. 메인 로직: 각 온도 데이터에 대해 조합 데이터셋 생성
for temp, ads_df in data_to_process.items():
    
    # 저장할 폴더 생성 (이미 있으면 그냥 넘어감)
    output_dir = f"Ar_NEW_{temp}"
    os.makedirs(output_dir, exist_ok=True)
    print(f"--- {temp} 데이터 처리 시작 ---")
    print(f"결과 저장 폴더: {output_dir}")

    # 1-1. 흡착 데이터와 메타 데이터 병합
    # how='inner'는 양쪽 데이터에 모두 'name'이 존재하는 경우만 남깁니다.
    merged_df = pd.merge(META, ads_df, on='name', how='inner')

    # 1-2. 저압-고압 모든 조합에 대해 파일 생성
    for low_p in low_pressure_list:
        for high_p in high_pressure_list:
            
            # 파일 이름 생성 (EX 파일 형식과 유사하게)
            # 예: Ar_273K_from_0.01_to_15.0_dataset.csv
            filename = f"Ar_{temp}_from_{low_p}_to_{high_p}_dataset.csv"
            filepath = os.path.join(output_dir, filename)

            # 필요한 데이터만으로 새 데이터프레임 생성
            # 먼저 구조 정보 컬럼들을 선택
            # 'name' 컬럼은 META와 ads_df에 모두 있으므로 META의 컬럼 리스트를 사용
            dataset_df = merged_df[META.columns].copy()
            
            # Input과 Output 컬럼 추가
            dataset_df['Input'] = merged_df[low_p]
            dataset_df['Output'] = merged_df[high_p]
            
            # CSV 파일로 저장 (인덱스는 저장하지 않음)
            dataset_df.to_csv(filepath, index=False)
            
    print(f"총 {len(low_pressure_list) * len(high_pressure_list)}개의 조합 데이터셋 생성을 완료했습니다.\n")

print("모든 작업이 완료되었습니다.")

--- 273K 데이터 처리 시작 ---
결과 저장 폴더: Ar_NEW_273K
총 15개의 조합 데이터셋 생성을 완료했습니다.

--- 313K 데이터 처리 시작 ---
결과 저장 폴더: Ar_NEW_313K
총 15개의 조합 데이터셋 생성을 완료했습니다.

모든 작업이 완료되었습니다.


In [15]:
import os 
import pandas as pd 

# Ar에 대한 데이터
final_adsorption_data_273K = pd.read_csv("./final_adsorption_data_273K.csv")
final_adsorption_data_313K = pd.read_csv("./final_adsorption_data_313K.csv")

# 메타 데이터 로딩
META = pd.read_csv("../2019-11-01-ASR-public_12020.csv")[['filename', 'LCD', 'PLD', 'LFPD', 'cm3_g', 'ASA_m2_cm3', 'ASA_m2_g',
       'NASA_m2_cm3', 'NASA_m2_g', 'AV_VF', 'AV_cm3_g', 'NAV_cm3_g',
       'Has_OMS',]]

# --- 코드 수정 시작 ---

# 0. 기본 설정 및 전처리
# META의 Has_OMS 라벨 인코딩
META['Has_OMS'] = META['Has_OMS'].map({'Yes':1, 'No':0})
# META 데이터의 'filename' 컬럼을 흡착 데이터와 병합하기 위해 'name'으로 변경
META.rename(columns={'filename': 'name'}, inplace=True)

# 압력 구간 리스트 정의
low_pressure_list = ['henry_coeff', '0.01', '0.05', '0.1', '0.5']
high_pressure_list = ['1.0', '5.0', '15.0']

# 처리할 온도별 데이터 딕셔너리
data_to_process = {
    '273K': final_adsorption_data_273K,
    '313K': final_adsorption_data_313K
}

# 파일 이름에 사용할 문자열을 변환하는 헬퍼 함수
def format_pressure_for_filename(p_str):
    """파일 이름 형식에 맞게 문자열을 변환합니다."""
    if p_str == 'henry_coeff':
        return 'Henry'
    # '.0'으로 끝나는 문자열(예: '1.0')에서 '.0'을 제거합니다.
    if p_str.endswith('.0'):
        return p_str[:-2]
    return p_str

# 1. 메인 로직: 각 온도 데이터에 대해 조합 데이터셋 생성
for temp, ads_df in data_to_process.items():
    
    # 저장할 폴더 생성
    output_dir = f"Ar_NEW_{temp}"
    os.makedirs(output_dir, exist_ok=True)
    print(f"--- {temp} 데이터 처리 시작 ---")
    print(f"결과 저장 폴더: {output_dir}")

    # 1-1. 데이터 병합
    merged_df = pd.merge(META, ads_df, on='name', how='inner')

    # 1-2. 저압-고압 모든 조합에 대해 파일 생성
    for low_p in low_pressure_list:
        for high_p in high_pressure_list:
            
            # 파일 이름용 문자열 변환
            low_p_name = format_pressure_for_filename(low_p)
            high_p_name = format_pressure_for_filename(high_p)
            
            # **수정된 파일 이름 생성 규칙**
            # 예: Ar_273K_Ar_273_0.01_to_Ar_273_15_dataset.csv
            filename = f"Ar_{temp}_Ar_{temp}_{low_p_name}_to_Ar_{temp}_{high_p_name}_dataset.csv"
            filepath = os.path.join(output_dir, filename)

            # 새 데이터프레임 생성
            # META.columns를 사용하면 'name' 컬럼이 포함된 구조 정보 전체를 가져올 수 있습니다.
            dataset_df = merged_df[META.columns].copy()
            
            # Input과 Output 컬럼 추가
            dataset_df['Input'] = merged_df[low_p]
            dataset_df['Output'] = merged_df[high_p]
            
            # CSV 파일로 저장
            dataset_df.to_csv(filepath, index=False)
            
    print(f"총 {len(low_pressure_list) * len(high_pressure_list)}개의 조합 데이터셋 생성을 완료했습니다.\n")

print("모든 작업이 완료되었습니다.")

--- 273K 데이터 처리 시작 ---
결과 저장 폴더: Ar_NEW_273K
총 15개의 조합 데이터셋 생성을 완료했습니다.

--- 313K 데이터 처리 시작 ---
결과 저장 폴더: Ar_NEW_313K
총 15개의 조합 데이터셋 생성을 완료했습니다.

모든 작업이 완료되었습니다.
